# Qualitative text generation

This notebook shows how to to generate qualitative documentation content directly from the ValidMind library. You will learn how to provide a custom prompt, optionally scope generation to specific document sections for context, and populate documentation text without manually writing markdown or HTML.

By the end, you will see how AI-assisted text generation can be used to draft an entire customer churn document from within a notebook, bringing the library experience in line with the qualitative text generation available in the UI.

<a id='toc2__'></a>

## Setting up

<a id='toc2_1__'></a>

### Install the ValidMind Library

To install the library:

In [ ]:
%pip install -q validmind

<a id='toc2_2__'></a>

### Initialize the ValidMind Library

<a id='toc2_2_1__'></a>

#### Register sample model

Let's first register a sample model for use with this notebook:

1. In a browser, [log in to ValidMind](https://docs.validmind.ai/guide/configuration/log-in-to-validmind.html).

2. In the left sidebar, navigate to **Inventory** and click **+ Register Model**.

3. Enter the model details and click **Next >** to continue to assignment of model stakeholders. ([Need more help?](https://docs.validmind.ai/guide/model-inventory/register-models-in-inventory.html))

4. Select your own name under the **MODEL OWNER** drop-down.

5. Click **Register Model** to add the model to your inventory.

<a id='toc2_2_2__'></a>

#### Apply documentation template

Once you've registered your model, let's select a documentation template. A template predefines sections for your model documentation and provides a general outline to follow, making the documentation process much easier.

1. In the left sidebar that appears for your model, click **Documents** and select **Development**.

2. Under **TEMPLATE**, select `Binary classification`.

3. Click **Use Template** to apply the template.

<a id='toc2_2_3__'></a>

#### Get your code snippet

Initialize the ValidMind Library with the *code snippet* unique to each model per document, ensuring your test results are uploaded to the correct model and automatically populated in the right document in the ValidMind Platform when you run this notebook.

1. On the left sidebar that appears for your model, select **Getting Started** and select `Development` from the **DOCUMENT** drop-down menu.
2. Click **Copy snippet to clipboard**.
3. Next, [load your model identifier credentials from an `.env` file](https://docs.validmind.ai/developer/model-documentation/store-credentials-in-env-file.html) or replace the placeholder with your own code snippet:

In [ ]:
# Load your model identifier credentials from an `.env` file

%load_ext dotenv
%dotenv .env

# Or replace with your code snippet

import validmind as vm

vm.init(
    api_host="http://localhost:5000/api/v1/tracking",
    api_key="..",
    api_secret="..",
    document="documentation", # requires library >=2.12.0
    model="..",
)

In [ ]:
%matplotlib inline

import xgboost as xgb
from validmind.utils import display

<a id='toc2_3__'></a>

### Preview the documentation template

Let's verify that you have connected the ValidMind Library to the ValidMind Platform and that the appropriate *template* is selected for your model.

You will upload documentation and test results unique to your model based on this template later on. For now, **take a look at the default structure that the template provides with [the `vm.preview_template()` function](https://docs.validmind.ai/validmind/validmind.html#preview_template)** from the ValidMind library and note the empty sections:

In [ ]:
vm.preview_template()

<a id='toc3__'></a>

## Load the Demo Dataset

In [ ]:
# You can also import taiwan_credit like this:
# from validmind.datasets.classification import taiwan_credit as demo_dataset
from validmind.datasets.classification import customer_churn as demo_dataset

df = demo_dataset.load_data()

<a id='toc3_1__'></a>

### Prepocess the raw dataset

In [ ]:
train_df, validation_df, test_df = demo_dataset.preprocess(df)

<a id='toc4__'></a>

## Train a model for testing

We train a simple customer churn model for our test.

In [ ]:
x_train = train_df.drop(demo_dataset.target_column, axis=1)
y_train = train_df[demo_dataset.target_column]
x_val = validation_df.drop(demo_dataset.target_column, axis=1)
y_val = validation_df[demo_dataset.target_column]

model = xgb.XGBClassifier(early_stopping_rounds=10)
model.set_params(
    eval_metric=["error", "logloss", "auc"],
)
model.fit(
    x_train,
    y_train,
    eval_set=[(x_val, y_val)],
    verbose=False,
)

<a id='toc5__'></a>

## Initialize ValidMind objects

We initize the objects required to run test suites using the ValidMind Library.

In [ ]:
vm_dataset = vm.init_dataset(
    input_id="raw_dataset",
    dataset=df,
    target_column=demo_dataset.target_column,
    class_labels=demo_dataset.class_labels,
)

vm_train_ds = vm.init_dataset(
    input_id="train_dataset",
    dataset=train_df,
    type="generic",
    target_column=demo_dataset.target_column,
)

vm_test_ds = vm.init_dataset(
    input_id="test_dataset",
    dataset=test_df,
    type="generic",
    target_column=demo_dataset.target_column,
)

vm_model = vm.init_model(model, input_id="model")

<a id='toc5_1__'></a>

### Assign predictions to the datasets

We can now use the `assign_predictions()` method from the `Dataset` object to link existing predictions to any model. If no prediction values are passed, the method will compute predictions automatically:

In [ ]:
vm_train_ds.assign_predictions(
    model=vm_model,
)
vm_test_ds.assign_predictions(
    model=vm_model,
)

## Log text in Markdown or HTML

In this section, we focus on using `vm.log_text()` with the current supported input formats. The examples show how to log qualitative text by passing either Markdown or HTML content and update an existing text block in the model documentation

In [ ]:
dataset_summary_text = """
## Dataset Summary

The customer churn dataset contains customer-level subscription, billing, and service usage attributes used to support model development and evaluation.

### Key Characteristics
- Records represent individual customers.
- Features include demographic, account, and service-related variables.
- The target captures whether a customer churned.

### Intended Use
This dataset is used to train and assess a churn prediction model that helps identify customers at higher risk of attrition.
"""

html =vm.log_text(
    content_id="dataset_summary_text",
    text=dataset_summary_text,
)

display(html)

In [ ]:
dataset_summary_text = """
<h2>Dataset Summary</h2>
<p>
The customer churn dataset contains customer-level subscription, billing, and service
usage attributes used to support model development and evaluation.
</p>
<h3>Key Characteristics</h3>
<ul>
  <li>Records represent individual customers.</li>
  <li>Features include demographic, account, and service-related variables.</li>
  <li>The target captures whether a customer churned.</li>
</ul>
<h3>Intended Use</h3>
<p>
This dataset is used to train and assess a churn prediction model that helps identify
customers at higher risk of attrition.
</p>
"""
html = vm.log_text(
    content_id="dataset_summary_text",
    text=dataset_summary_text,
)
display(html)

## Generate text with default settings

In this section, we focus on generating qualitative text with the default behavior of `vm.log_text()`. When no additional configuration is provided, ValidMind uses the current document as context to generate content and update the corresponding section in the model documentation.

In [ ]:
result = vm.run_text_generation(
    content_id="dataset_summary_text",
)
result

In [ ]:
result.log()

## Generate text with a custom prompt

In this section, we focus on generating qualitative text using a custom prompt. This allows you to guide both what content should be generated and why it should be written in a particular way for the model documentation.

In [ ]:
result = vm.run_text_generation(
    content_id="dataset_summary_text",
    prompt="Generate a summary of the dataset using bullet points.",
)
result

In [ ]:
result.prompt

In [ ]:
result.log()

## Generate text with contextual content IDs

In this section, we focus on generating qualitative text using specific content_ids as context. This allows you to guide the generation with selected parts of the document instead of relying on the full document context

In [ ]:
result = vm.run_text_generation(
    content_id="dataset_summary_text",
    context={"content_ids": ["validmind.data_validation.DescriptiveStatistics"]},
)

In [ ]:
result.context

In [ ]:
result.log()

In [ ]:
section_ids = ["data_description", "model_development"]
content_ids = vm.get_content_ids(section_ids)
content_ids

In [ ]:
result = vm.run_text_generation(
    content_id="dataset_summary_text",
    context={"content_ids": content_ids},
)

In [ ]:
result.context

In [ ]:
result.log()

<!-- VALIDMIND COPYRIGHT -->

<small>

***

Copyright © 2023-2026 ValidMind Inc. All rights reserved.<br>
Refer to [LICENSE](https://github.com/validmind/validmind-library/blob/main/LICENSE) for details.<br>
SPDX-License-Identifier: AGPL-3.0 AND ValidMind Commercial</small>